# A07 · Reasoning Models & Test-Time Compute

**Track:** Advanced / LLM engineering (lesson 7 of 10) · **Time:** ~3 h · **Mode:** runs fully OFFLINE (NumPy simulation, CPU, ~1 min); no API key needed
**Prereqs:** A02 (prompting, chain-of-thought), A04 (evals), A06 (verifiable rewards, reward hacking), basic probability (binomial) · **You will build:** exact and simulated voting-accuracy curves, a best-of-N selector with an imperfect (and hackable) verifier, a sequential-vs-parallel compute simulator, a verifiable task suite with checkers, an adaptive-stopping sampler and a budget planner

---
## 🎯 Why this matters
For years, "making the model better" meant a bigger model or more training. Reasoning models add a second axis: **spend more compute at inference time** on a hard question - think longer, sample several attempts, check the work. That is a cost/latency/accuracy trade-off you, the engineer, now control per request (a "thinking budget" or "effort" setting) and it can be worth more than a model upgrade on the right tasks and a pure waste on the wrong ones.

**Real scenario.** *A finance-operations team* has an assistant that reconciles invoices: multi-step arithmetic over line items, tax and discounts, plus a few logic checks on approval rules. A single answer is right about 75% of the time. Options: (a) allow longer chains of thought, (b) sample 5-20 answers and take the majority, (c) sample and let a checker pick, (d) a bigger model. Which buys the most accuracy per dollar *for which kind of question*, how do you know when the extra compute has stopped helping, and how can a verifier go wrong?

## 🧭 Learning objectives
By the end you can:
1. Derive and compute **majority-vote (self-consistency) accuracy**, and say exactly when voting helps, saturates, or hurts.
2. Model **best-of-N with an imperfect verifier** and show how a hackable verifier makes more samples *worse*.
3. Compare **parallel** (many samples) and **sequential** (longer thinking) scaling and identify which bottleneck each fixes.
4. Build a small **verifiable task suite** with programmatic checkers and measure accuracy-vs-cost on it, including an adaptive stopping rule.
5. Decide when thinking helps, and set a budget with a simple planning procedure.

## 1 · Concept: three ways to turn compute into accuracy

```
 parallel      ┌─▶ sample 1 ─┐
 (width)   x ──┼─▶ sample 2 ─┼─▶ aggregate: majority vote | verifier picks | reward-model picks ─▶ answer
               └─▶ sample N ─┘

 sequential    x ─▶ think ─▶ think more ─▶ check ─▶ revise ─▶ ... ─▶ answer     (longer chain; "extended thinking")
 (depth)

 search        tree/beam over partial reasoning steps, scored by a verifier / value model (needs a step-level scorer)
```

* **Reasoning models** are trained (typically with RL on verifiable rewards, A06) to produce a long internal chain of thought before answering, and expose a knob for *how much*: a token **budget** for thinking, or a coarser **effort** level. Check your provider's current docs for the exact parameters; they change.
* **Self-consistency** (Wang et al., 2022): sample several chains of thought at temperature > 0 and take the most frequent *final answer*. It works because different wrong chains tend to disagree while right chains agree.
* **Best-of-N with a verifier**: sample N, score each with a checker (unit tests, a calculator, a learned reward model, an LLM judge), return the best. Quality depends entirely on the verifier.
* **Cost model.** Output (and thinking) tokens dominate: cost $\approx N\times(\text{tokens per attempt})\times$ price, and parallel samples share latency but not cost; a longer sequential chain costs the same tokens but *adds* latency. We use "tokens" as the unit throughout (convert with your own price list).

## 2 · Build it: the mathematics of majority voting
**Binary case.** Suppose each independent attempt is correct with probability $p$, otherwise it gives *the same wrong answer* (worst case for voting). With $N$ odd samples the vote is correct iff more than half are correct:

$$\mathrm{Maj}(p,N)=\sum_{k>N/2}\binom Nk p^k(1-p)^{N-k}.$$

This is the Condorcet jury theorem: if $p>1/2$ accuracy $\to1$ as $N\to\infty$; if $p<1/2$ it $\to0$ - **voting amplifies whatever the majority tendency is, including a wrong one**. A Hoeffding bound gives the rate: $P(\text{vote wrong})\le\exp\!\big(-2N(p-\tfrac12)^2\big)$.

**Many wrong answers.** In practice wrong answers *disagree* with each other. If the correct answer has probability $p$ and the wrong mass is spread over $m$ distinct answers, the correct answer only needs a **plurality**, which can happen even when $p<1/2$ (e.g. $p=0.3$ against ten wrong answers at 7% each). We estimate it by simulation (draw a multinomial of counts, check the winner).

In [ ]:
import numpy as np, math, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def maj_binary(p, N):
    """Exact majority-vote accuracy, odd N, p scalar or array."""
    p = np.asarray(p, dtype=float)[..., None]; k = np.arange(N // 2 + 1, N + 1)
    logc = np.array([math.lgamma(N + 1) - math.lgamma(j + 1) - math.lgamma(N - j + 1) for j in k])
    return np.exp(logc + k * np.log(np.clip(p, 1e-300, 1)) + (N - k) * np.log(np.clip(1 - p, 1e-300, 1))).sum(-1)

def multinomial_rows(rng, N, probs):
    try: return rng.multinomial(N, probs)
    except Exception: return np.array([rng.multinomial(N, p) for p in probs])      # older NumPy: no 2-D pvals

def plurality_acc(p, N, m, rng, trials=1):
    """Plurality-vote accuracy; p array of per-problem success probs; wrong mass spread uniformly over m wrong answers.
    Ties are credited fractionally (a uniformly random tie-break)."""
    p = np.atleast_1d(np.asarray(p, float)); probs = np.concatenate([p[:, None], np.repeat(((1 - p) / m)[:, None], m, 1)], 1)
    out = 0.0
    for _ in range(trials):
        C = multinomial_rows(rng, N, probs); mx = C.max(1); tied = (C == mx[:, None]).sum(1)
        out += np.mean((C[:, 0] == mx) / tied)
    return out / trials

rng = np.random.default_rng(0)
Ns = np.arange(1, 66, 2)
curves = {"binary, p=0.6": [float(maj_binary(0.6, n)) for n in Ns],
          "binary, p=0.4": [float(maj_binary(0.4, n)) for n in Ns],
          "plurality, p=0.3, 10 wrong answers": [plurality_acc(np.full(4000, 0.3), n, 10, rng) for n in Ns]}
plt.figure(figsize=(6.5, 3.8))
for k, v in curves.items(): plt.plot(Ns, v, label=k)
plt.axhline(0.5, color="grey", lw=0.5); plt.xlabel("number of samples N"); plt.ylabel("accuracy of the vote"); plt.title("self-consistency: amplifies the majority tendency"); plt.legend(); plt.show()

# --- checks: exact formula vs Monte Carlo, Hoeffding bound, the three regimes
mc = np.mean(np.random.default_rng(1).binomial(15, 0.6, 200_000) > 7)
assert abs(float(maj_binary(0.6, 15)) - mc) < 0.004, (maj_binary(0.6, 15), mc)
assert abs(float(maj_binary(0.5, 11)) - 0.5) < 1e-9, "p = 1/2 is the knife edge: a coin-flip model stays a coin flip"
assert all(1 - float(maj_binary(0.6, n)) <= math.exp(-2 * n * 0.1 ** 2) + 1e-12 for n in Ns), "Hoeffding upper-bounds the error"
assert curves["binary, p=0.6"][-1] > 0.9 and curves["binary, p=0.4"][-1] < 0.1, "binary vote amplifies p>1/2 up and p<1/2 down"
assert curves["plurality, p=0.3, 10 wrong answers"][-1] > 0.9, "plurality can rescue p<1/2 when wrong answers are spread out"
print(f"N=1 -> N=65:  p=0.6 binary {curves['binary, p=0.6'][0]:.2f} -> {curves['binary, p=0.6'][-1]:.2f};  p=0.4 binary {curves['binary, p=0.4'][0]:.2f} -> {curves['binary, p=0.4'][-1]:.2f};  p=0.3 plurality {curves['plurality, p=0.3, 10 wrong answers'][0]:.2f} -> {curves['plurality, p=0.3, 10 wrong answers'][-1]:.2f}")

### The catch: problems are not equally hard
The curves above assume one $p$ for every problem. Real question sets are *heterogeneous*: some are easy ($p_i\approx0.95$), some hopeless ($p_i\approx0.1$). Voting acts **per problem**, so overall accuracy is $\mathbb E_i[\mathrm{Maj}(p_i,N)]$ - which is *not* $\mathrm{Maj}(\bar p,N)$. Voting helps most on problems with $p_i$ just above the threshold and does nothing for problems the model basically cannot do. The ceiling as $N\to\infty$ is the fraction of problems with $p_i$ above the threshold (in the binary worst case, $P(p_i>1/2)$). Let us compare a homogeneous model and a heterogeneous one with the *same mean* accuracy 0.6.

In [ ]:
rg = np.random.default_rng(3)
p_i = rg.beta(1.2, 0.8, 20000); p_i = p_i * 0.6 / p_i.mean()          # heterogeneous problem set with mean exactly 0.6
p_i = np.clip(p_i, 0, 1); print("mean p =", round(float(p_i.mean()), 3), "  fraction of problems with p_i > 1/2:", round(float((p_i > 0.5).mean()), 3))
Ns2 = np.array([1, 3, 5, 9, 17, 33, 65, 129])
homog = [float(maj_binary(0.6, n)) for n in Ns2]
heter = [float(maj_binary(p_i, n).mean()) for n in Ns2]
plt.figure(figsize=(6, 3.6)); plt.semilogx(Ns2, homog, "o-", label="homogeneous p=0.6"); plt.semilogx(Ns2, heter, "s-", label="heterogeneous, mean p=0.6")
plt.axhline((p_i > 0.5).mean(), color="grey", ls=":", label="ceiling = P(p_i > 1/2)"); plt.xlabel("N"); plt.ylabel("vote accuracy"); plt.legend(); plt.title("same average accuracy, very different returns"); plt.show()
print("homogeneous:", np.round(homog, 3)); print("heterogeneous:", np.round(heter, 3))
assert heter[-1] < homog[-1] - 0.05, "heterogeneity lowers the returns to voting"
assert abs(heter[-1] - (p_i > 0.5).mean()) < 0.03, "as N grows the vote accuracy approaches the fraction of problems with p_i above 1/2"
assert heter[-1] > heter[0] and heter[-1] - heter[0] < 0.1, "voting helps only a little: the gain is capped by P(p_i > 1/2)"

In this example (binary worst case) 129 votes lift the heterogeneous model only from 0.60 to about 0.635, versus 0.99 for the homogeneous model with the same average - a 2-point-of-ceiling lesson in why per-problem pass rates matter. **Practical lesson:** *average* single-sample accuracy does not tell you what voting will buy you; the **distribution** of per-problem pass rates does. Estimate it: sample e.g. 8-16 answers on a few hundred eval questions and histogram the per-question pass rates before committing to a voting budget.

## 3 · Build it: best-of-N with an imperfect verifier
Now each sample is scored by a verifier and we return the highest-scoring sample. Model the verifier's score as Gaussian: correct answers score $\mathcal N(\mu,1)$, wrong ones $\mathcal N(0,1)$; $\mu$ (the "d-prime") measures verifier quality. Upper bound for any selector: **pass@N** $=1-(1-p)^N$ (at least one sample is right). $\mu=0$ is a useless verifier (accuracy stays $p$); $\mu\to\infty$ is perfect (accuracy $\to$ pass@N).

Then the realistic failure: a fraction of the *wrong* answers are **confidently wrong** (fluent, well-formatted, convincing) and score *higher* than typical correct answers - exactly the reward-hacking pattern of A06. With more samples you find more of them.

In [ ]:
def best_of_n(p, N, mu, f_hard=0.0, mu_hard=3.0, trials=20000, seed=0):
    r = np.random.default_rng(seed)
    ok = r.uniform(size=(trials, N)) < p
    hard = r.uniform(size=(trials, N)) < f_hard                      # confidently-wrong subpopulation (only among wrong answers)
    score = np.where(ok, r.normal(mu, 1, (trials, N)), np.where(hard, r.normal(mu_hard, 1, (trials, N)), r.normal(0, 1, (trials, N))))
    return float(ok[np.arange(trials), score.argmax(1)].mean())

p0 = 0.4; Nb = [1, 2, 4, 8, 16, 32, 64, 128, 256]
res = {"pass@N (upper bound)": [1 - (1 - p0) ** n for n in Nb]}
for mu in [0.0, 1.0, 2.0, 4.0]: res[f"verifier d'={mu:g}"] = [best_of_n(p0, n, mu) for n in Nb]
res["d'=2 but 10% of wrong answers are 'persuasive'"] = [best_of_n(p0, n, 2.0, f_hard=0.10) for n in Nb]
plt.figure(figsize=(7, 4))
for k, v in res.items(): plt.semilogx(Nb, v, "o-" if "pass" not in k else "k--", label=k)
plt.xlabel("N samples"); plt.ylabel("accuracy of selected answer"); plt.title("best-of-N: only as good as the verifier"); plt.legend(fontsize=7); plt.show()
for k, v in res.items(): print(f"{k:<46}", np.round(v, 3))

assert abs(res["verifier d'=0"][-1] - p0) < 0.02, "a useless verifier cannot beat a single sample"
assert all(a <= b + 0.01 for a, b in zip(res["verifier d'=2"], res["pass@N (upper bound)"])), "no selector beats pass@N"
assert res["verifier d'=4"][3] > res["verifier d'=2"][3] > res["verifier d'=1"][3] > res["verifier d'=0"][3], "better verifier -> better accuracy at fixed N"
bad = res["d'=2 but 10% of wrong answers are 'persuasive'"]
peak = int(np.argmax(bad)); print(f"\nflawed verifier: accuracy peaks at N={Nb[peak]} ({bad[peak]:.2f}) and falls to {bad[-1]:.2f} at N={Nb[-1]} - below a single sample ({bad[0]:.2f}).")
assert 0 < peak < len(Nb) - 1 and bad[-1] < bad[peak] - 0.3 and bad[-1] < bad[0], "more samples + a hackable verifier = worse answers"

**Why it falls.** With $N$ samples the *maximum* of the "persuasive wrong" scores grows with $\log N$ and eventually exceeds the maximum over the (fewer) correct answers: you are running an optimiser against the verifier, i.e. Goodhart's law again. Three defences: (1) prefer **programmatic verifiers** (tests, calculators, schema/constraint checks); (2) cap $N$ at the point where a gold eval says returns stop (here $N\approx8$); (3) combine signals - e.g. vote *among the samples the verifier accepts* (below, section 5).

## 4 · Build it: sequential vs parallel scaling
Two different bottlenecks:
* **Depth-limited** problems: the chain must be long enough to *contain* the solution (multi-step arithmetic, planning). A short chain fails no matter how many times you resample it.
* **Variance-limited** problems: the model *can* do it but is unreliable (a slip somewhere); resampling and voting averages the luck away.

We simulate both. Depth-limited: problem $i$ needs about $d_i$ thinking tokens (log-normal, median 800); a chain with budget $T$ succeeds with probability $p_i(T)=p_{max}\,\sigma\!\big(k(\ln T-\ln d_i)\big)$ (a smooth capability threshold; $p_{max}=0.95$, $k=2.5$). **These are modelling assumptions chosen to be plausible, not measured laws** - the point is the *shape of the trade-off*, which you must then measure on your task. Variance-limited: per-problem pass rate drawn from a Beta distribution *independent of $T$*. Parallel strategy: $N$ chains of budget $T_0$, plurality vote over 8 possible wrong answers. Cost = total tokens.

In [ ]:
rs = np.random.default_rng(7); P = 3000
d_i = np.exp(rs.normal(np.log(800), 1.0, P))
p_depth = lambda T, pmax=0.95, k=2.5: pmax * sigmoid(k * (np.log(T) - np.log(d_i)))
p_var = np.clip(rs.beta(2.5, 1.5, P) * 0.95, 0, 1)                              # mean about 0.59, independent of T

seqT = np.array([200, 400, 800, 1600, 3200, 6400, 12800, 25600])
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
out = {}
for a, (name, pf) in zip(ax, [("depth-limited", p_depth), ("variance-limited", lambda T: p_var)]):
    seq = [float(pf(T).mean()) for T in seqT]; out[name] = {"seq": seq}
    a.loglog(seqT, seq, "ko-", label="sequential: one chain, budget T")
    for T0 in [800, 1600]:
        Nl = [1, 3, 9, 27, 81]; par = [plurality_acc(pf(T0), n, 8, rs) for n in Nl]; out[name][T0] = (Nl, par)
        a.loglog([n * T0 for n in Nl], par, "s--", label=f"parallel: N chains of {T0} tokens, vote")
    a.set_title(name); a.set_xlabel("total tokens spent per question"); a.grid(alpha=0.3)
ax[0].set_ylabel("accuracy"); ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

# at the same total cost (3200 tokens): one 3200-token chain vs 4 x 800-token chains vs 2 x 1600
cost = 3200
for name, pf in [("depth-limited", p_depth), ("variance-limited", lambda T: p_var)]:
    s = float(pf(cost).mean()); par = plurality_acc(pf(800), 4, 8, rs); par2 = plurality_acc(pf(1600), 2, 8, rs)
    print(f"{name:<17} same {cost} tokens: sequential {s:.3f} | 4x800 voted {par:.3f} | 2x1600 voted {par2:.3f}")
sd, pd_ = float(p_depth(cost).mean()), plurality_acc(p_depth(800), 4, 8, rs)
sv, pv = float(p_var.mean()), plurality_acc(p_var, 4, 8, rs)
assert sd > pd_ + 0.1, "depth-limited: spend the tokens on a longer chain"
assert pv > sv + 0.02, "variance-limited: spend the tokens on independent samples"
assert out["depth-limited"]["seq"][-1] < 0.96 and out["depth-limited"]["seq"][-1] - out["depth-limited"]["seq"][-2] < 0.02, "sequential saturates at p_max"

**Reading the two panels.** When the bottleneck is depth, parallel voting over short chains is a poor substitute for a longer chain (the short chains simply cannot reach the answer); when it is variance, longer chains buy nothing (curve is flat) and sampling wins. Most real tasks are a **mixture** - which is why the winning production recipe is usually: *give each chain enough budget to be able to solve the problem, then use parallel samples + a checker for the residual unreliability*. And past the saturation point (flat tail) additional tokens are pure cost.

### Budget planner
Given a target accuracy, choose the cheapest $(T_0,N)$ from a grid. This is the same procedure you would run on measured data (replace the simulated pass rates by pass rates estimated on your eval set per budget).

In [ ]:
grid_T = [400, 800, 1600, 3200, 6400]; grid_N = [1, 3, 5, 9, 17, 33]
table = {(T0, n): plurality_acc(p_depth(T0), n, 8, np.random.default_rng(T0 + n), trials=2) for T0 in grid_T for n in grid_N}
def cheapest(target):
    ok = [(T0 * n, T0, n, a) for (T0, n), a in table.items() if a >= target]
    return min(ok) if ok else None
for tgt in [0.6, 0.8, 0.9, 0.95]:
    c = cheapest(tgt); print(f"target {tgt:.2f}: " + (f"cheapest = N={c[2]} chains x {c[1]} tokens = {c[0]:,} tokens (acc {c[3]:.3f})" if c else "not reachable on this grid"))
c9 = cheapest(0.9); assert c9 is not None and all(a < 0.9 or T0 * n >= c9[0] for (T0, n), a in table.items())
assert cheapest(0.8)[0] < c9[0], "higher targets cost more"
c99 = cheapest(0.99); print("target 0.99:", c99)
assert c99 is None or c99[0] > 2 * c9[0], "the last points of accuracy are disproportionately expensive (or unreachable on this grid)"

## 5 · Use it: a verifiable task suite
Time to leave Gaussians and score real, checkable items. Our *finance-ops* suite has two task families, each with a **ground truth and a programmatic checker** (the property that makes RL and best-of-N work, A06):

1. **Ledger arithmetic**: a starting balance and $k$ sequential adjustments (deposits, withdrawals, multipliers); the answer is an integer. Difficulty = number of steps $k$. *Verifier*: re-execute the steps (a calculator tool).
2. **Approval-rule logic** (knights-and-knaves style): $n$ approvers each make a statement about the others (who is *reliable* (always right) or *unreliable* (always wrong)); find who is who. Exactly one assignment is consistent. *Verifier*: check a proposed assignment against all statements - **checking is much cheaper than solving**, the asymmetry verification relies on.

Answer format: the model must end with `Answer: <value>`; we need a robust **extractor + normaliser** (a classic source of fake failures: `1,200` vs `1200`).

In [ ]:
import re, itertools
def extract_answer(text):
    m = re.findall(r"Answer:\s*(.+)", text)
    if not m: return None
    a = m[-1].strip().strip(".").replace(",", "").replace("$", "").replace(" ", "")
    if re.fullmatch(r"-?\d+(\.0+)?", a): a = str(int(float(a)))
    return a.upper()

assert extract_answer("Let me compute... so the total is\nAnswer: $1,200.") == "1200"
assert extract_answer("Answer: 1200.0") == "1200" and extract_answer("Answer: rrk") == "RRK" and extract_answer("no final line") is None
assert extract_answer("Answer: 5\nWait, recheck.\nAnswer: 7") == "7", "last answer line wins"

# ---------- family 1: ledger arithmetic
OPS = {"add": ("{n} units are received", lambda v, n: v + n), "sub": ("{n} units are shipped out", lambda v, n: v - n), "mul": ("stock is multiplied by {n} after a bulk import", lambda v, n: v * n)}
def make_arith(rng, k):
    v = start = int(rng.integers(20, 200)); steps = []
    for _ in range(k):
        op = str(rng.choice(["add", "sub", "mul"], p=[0.4, 0.4, 0.2])); n = int(rng.integers(2, 4)) if op == "mul" else int(rng.integers(5, 60))
        if op == "sub" and n >= v: op = "add"
        v = OPS[op][1](v, n); steps.append((op, n))
    text = f"A warehouse holds {start} crates. In order: " + "; ".join(f"({i+1}) " + OPS[o][0].format(n=n) for i, (o, n) in enumerate(steps)) + ". How many crates are there at the end?"
    return {"kind": "arith", "k": k, "start": start, "steps": steps, "text": text, "truth": str(v)}
def verify_arith(prob, ans):                                     # re-execution verifier
    v = prob["start"]
    for op, n in prob["steps"]: v = OPS[op][1](v, n)
    return ans == str(v)

# ---------- family 2: approvers (reliable R / unreliable U)
NAMES = "ABCDEF"
def stmt_true(stmt, t):                                          # t: tuple of booleans, True = reliable
    kind, x, y = stmt
    return {"rel": t[x], "unrel": not t[x], "same": t[x] == t[y], "diff": t[x] != t[y]}[kind]
def stmt_text(stmt):
    kind, x, y = stmt
    return {"rel": f"{NAMES[x]} is reliable", "unrel": f"{NAMES[x]} is unreliable", "same": f"{NAMES[x]} and {NAMES[y]} are the same type", "diff": f"{NAMES[x]} and {NAMES[y]} are different types"}[kind]
def consistent(stmts, t): return all(stmt_true(s, t) == t[i] for i, s in enumerate(stmts))     # reliable people state truths; unreliable state falsehoods
def make_logic(rng, n):
    while True:
        stmts = []
        for i in range(n):
            kind = str(rng.choice(["rel", "unrel", "same", "diff"])); others = [j for j in range(n) if j != i]
            x = int(rng.choice(others)); y = int(rng.choice([j for j in others if j != x] or others)); stmts.append((kind, x, y))
        sols = [t for t in itertools.product([True, False], repeat=n) if consistent(stmts, t)]
        if len(sols) == 1:
            truth = "".join("R" if b else "U" for b in sols[0])
            text = "Approvers " + ", ".join(NAMES[:n]) + " are each either reliable (always truthful) or unreliable (always wrong). " + " ".join(f"{NAMES[i]} says: '{stmt_text(s)}.'" for i, s in enumerate(stmts)) + " Who is reliable? Answer as a string of R/U in order."
            return {"kind": "logic", "n": n, "stmts": stmts, "text": text, "truth": truth, "trap": bool(rng.uniform() < 0.25), "tempting": int(rng.integers(n))}
def verify_logic(prob, ans):
    if ans is None or len(ans) != prob["n"] or set(ans) - {"R", "U"}: return False
    return consistent(prob["stmts"], tuple(c == "R" for c in ans))

rg = np.random.default_rng(42)
demo_a, demo_l = make_arith(rg, 4), make_logic(rg, 4)
print(demo_a["text"], "->", demo_a["truth"]); print(demo_l["text"], "->", demo_l["truth"])
assert verify_arith(demo_a, demo_a["truth"]) and not verify_arith(demo_a, str(int(demo_a["truth"]) + 1))
assert verify_logic(demo_l, demo_l["truth"]) and not verify_logic(demo_l, "R" * 4 if demo_l["truth"] != "R" * 4 else "U" * 4)
assert sum(verify_logic(demo_l, "".join(a)) for a in itertools.product("RU", repeat=4)) == 1, "unique-solution puzzles: exactly one assignment passes the checker"
SUITE = [make_arith(rg, int(k)) for k in rg.integers(2, 11, 160)] + [make_logic(rg, int(n)) for n in rg.integers(3, 6, 80)]
print(len(SUITE), "problems:", sum(p["kind"] == "arith" for p in SUITE), "arithmetic,", sum(p["kind"] == "logic" for p in SUITE), "logic")

### A simulated "model" with realistic error structure
We cannot call a model offline, so a stochastic *solver* plays one - but with error mechanics that mirror real chains of thought rather than a single coin flip:
* **Arithmetic:** each step slips at random with probability 0.04 (adds $\pm1$ or $\pm10$), *and* a multiplication step is systematically misread as an addition 12% of the time (a consistent misconception, so these wrong answers *cluster*).
* **Logic:** an attempt errs with probability $0.08(n-1)$ (more approvers, more chances), plus 0.30 on 'trap' puzzles (25% of them); 60% of errors flip one specific *tempting* label, so wrong answers concentrate on one alternative.
Expect accuracy to fall with difficulty (number of steps/approvers). Cost per attempt (illustrative tokens): arithmetic $60+45k$, logic $100+110n$.

In [ ]:
def sample_answer(prob, rng, eps_arith=0.04, misread_mul=0.12):
    if prob["kind"] == "arith":
        v = prob["start"]
        for op, n in prob["steps"]:
            if op == "mul" and rng.uniform() < misread_mul: v = v + n               # systematic misreading: "multiplied by n" read as "plus n"
            else: v = OPS[op][1](v, n)
            if rng.uniform() < eps_arith: v += int(rng.choice([-10, -1, 1, 10]))      # random slip
        text = f"...reasoning...\nAnswer: {v:,}"
    else:
        a = list(prob["truth"]); q = 0.08 * (prob["n"] - 1) + (0.30 if prob["trap"] else 0.0)     # chance this attempt errs
        if rng.uniform() < q:
            j = prob["tempting"] if rng.uniform() < 0.6 else int(rng.integers(prob["n"])); a[j] = "U" if a[j] == "R" else "R"
        text = "...reasoning...\nAnswer: " + "".join(a)
    return extract_answer(text)
tokens_per_attempt = lambda p: 60 + 45 * p["k"] if p["kind"] == "arith" else 100 + 110 * p["n"]
is_correct = lambda p, a: a == p["truth"]

NMAX = 33; rng_s = np.random.default_rng(100)
SAMPLES = [[sample_answer(p, rng_s) for _ in range(NMAX)] for p in SUITE]                  # sample once, evaluate any prefix
CORR = np.array([[is_correct(p, a) for a in s] for p, s in zip(SUITE, SAMPLES)])            # (problems, NMAX)

def bucket(p):
    d = p["k"] if p["kind"] == "arith" else p["n"] * 2
    return ("arith 2-4", "arith 5-7", "arith 8-10")[0 if d <= 4 else 1 if d <= 7 else 2] if p["kind"] == "arith" else f"logic n={p['n']}"
buckets = sorted({bucket(p) for p in SUITE})
for b in buckets:
    idx = [i for i, p in enumerate(SUITE) if bucket(p) == b]; print(f"{b:<12} n={len(idx):<4} single-sample accuracy {CORR[idx, 0].mean():.2f}   (mean over 33 samples: {CORR[idx].mean():.2f})")
assert CORR[[i for i, p in enumerate(SUITE) if bucket(p) == "arith 2-4"]].mean() > CORR[[i for i, p in enumerate(SUITE) if bucket(p) == "arith 8-10"]].mean() + 0.1, "harder -> lower pass rate"

Now the aggregators over the *same* samples: **majority@N** (plurality, ties broken by first occurrence), **pass@N** (upper bound), **verified@N** with a perfect programmatic verifier (return a verified answer if any, else the vote), and **noisy-verifier@N**: an LLM-judge-style checker that accepts correct answers with probability 0.90 and wrong ones with probability 0.25, then votes among the accepted ones (falling back to the overall vote). Note what each needs: the vote needs nothing; the verifier needs a checker.

In [ ]:
from collections import Counter
def vote(answers):
    c = Counter(a for a in answers if a is not None); 
    if not c: return None
    top = max(c.values()); return next(a for a in answers if a is not None and c[a] == top)       # first-occurring among the leaders

VERIFY = {"arith": verify_arith, "logic": verify_logic}
rv = np.random.default_rng(5)
NOISY = [[(rv.uniform() < (0.90 if is_correct(p, a) else 0.25)) for a in s] for p, s in zip(SUITE, SAMPLES)]

def agg(i, N, method):
    p, s = SUITE[i], SAMPLES[i][:N]
    if method == "single": return s[0]
    if method == "majority": return vote(s)
    if method == "pass": return p["truth"] if any(is_correct(p, a) for a in s) else s[0]
    if method == "verified":
        ok = [a for a in s if a is not None and VERIFY[p["kind"]](p, a)]; return ok[0] if ok else vote(s)
    if method == "noisy_verifier":
        acc = [a for a, k in zip(s, NOISY[i][:N]) if k]; return vote(acc) if acc else vote(s)

Nlist = [1, 3, 5, 9, 17, 33]; methods = ["majority", "verified", "noisy_verifier", "pass"]
R = {m: [np.mean([is_correct(SUITE[i], agg(i, n, m)) for i in range(len(SUITE))]) for n in Nlist] for m in methods}
avg_tokens = np.mean([tokens_per_attempt(p) for p in SUITE])
print(f"{'method':<16}" + "".join(f"N={n:<6}" for n in Nlist) + f"   (avg {avg_tokens:.0f} tokens/attempt)")
for m in methods: print(f"{m:<16}" + "".join(f"{v:<8.3f}" for v in R[m]))
plt.figure(figsize=(7, 4))
for m in methods: plt.plot(np.array(Nlist) * avg_tokens, R[m], "o-", label=m)
plt.xscale("log"); plt.xlabel("avg tokens spent per problem"); plt.ylabel("suite accuracy"); plt.legend(); plt.title("accuracy vs cost on the verifiable suite"); plt.show()

single = CORR[:, 0].mean()
assert abs(R["majority"][0] - single) < 1e-9 and R["majority"][-1] > single + 0.08, "voting improves on a single sample"
assert R["verified"][-1] >= R["majority"][-1] and R["verified"][-1] <= R["pass"][-1] + 1e-9, "perfect verifier sits between the vote and pass@N"
assert R["pass"][-1] > 0.97, "with 33 attempts almost every problem has at least one correct sample"
assert R["noisy_verifier"][-1] > R["majority"][-1] - 0.03 and R["noisy_verifier"][-1] < R["verified"][-1], "a noisy checker helps less than a real one"
gain = {m: R[m][-1] - single for m in methods}; print("gain over single sample at N=33:", {k: round(float(v), 3) for k, v in gain.items()})

**Observations to carry into practice.** (1) A cheap *programmatic* verifier is worth more than a large vote: it takes accuracy to nearly pass@N. (2) A noisy LLM-style verifier lands between the vote and the exact checker; how much it adds depends entirely on its false-accept/false-reject rates (here 25% / 10%), so measure those on labelled data before paying for it (A04). (3) The vote's gain concentrates in the mid-difficulty buckets; check by difficulty:

In [ ]:
print(f"{'bucket':<12}{'single':>8}{'maj@9':>8}{'maj@33':>8}{'verified@9':>12}")
byb = {}
for b in buckets:
    idx = [i for i, p in enumerate(SUITE) if bucket(p) == b]
    row = [np.mean([is_correct(SUITE[i], agg(i, n, m)) for i in idx]) for n, m in [(1, "majority"), (9, "majority"), (33, "majority"), (9, "verified")]]
    byb[b] = row; print(f"{b:<12}" + "".join(f"{v:>8.2f}" if j < 3 else f"{v:>12.2f}" for j, v in enumerate(row)))
assert all(r[2] >= r[0] - 0.02 for r in byb.values())

## 6 · Use it: adaptive compute - stop sampling when the answer is settled
Fixed $N$ wastes tokens on easy questions (all samples agree after 3) and under-serves hard ones. **Sequential stopping**: keep sampling until the leading answer is ahead of the runner-up by a margin $\Delta$ (or a cap $N_{max}$ is reached). Easy questions finish in $\Delta$ samples; contested ones keep going. This is the same idea as sequential hypothesis testing, and it is how you turn an "effort" knob into a *per-question* policy.

In [ ]:
def adaptive(i, margin=3, nmax=NMAX):
    c = Counter(); s = SAMPLES[i]
    for n in range(1, nmax + 1):
        c[s[n - 1]] += 1; top = c.most_common(2)
        if top[0][1] - (top[1][1] if len(top) > 1 else 0) >= margin: return top[0][0], n
    return vote(s[:nmax]), nmax

print(f"{'policy':<22}{'accuracy':>10}{'avg samples':>14}{'avg tokens':>12}")
fixed33 = np.mean([is_correct(SUITE[i], agg(i, 33, "majority")) for i in range(len(SUITE))]); toks33 = 33 * avg_tokens
print(f"{'fixed N=33':<22}{fixed33:>10.3f}{33:>14.1f}{toks33:>12.0f}")
pol = {}
for margin in [2, 3, 4]:
    outs = [adaptive(i, margin) for i in range(len(SUITE))]
    a = np.mean([is_correct(SUITE[i], o[0]) for i, o in enumerate(outs)]); ns = np.mean([o[1] for o in outs])
    t = np.mean([o[1] * tokens_per_attempt(SUITE[i]) for i, o in enumerate(outs)]); pol[margin] = (a, ns, t)
    print(f"{'adaptive margin='+str(margin):<22}{a:>10.3f}{ns:>14.1f}{t:>12.0f}")
a3, n3, t3 = pol[3]
assert a3 > fixed33 - 0.03, "adaptive stopping loses almost no accuracy..."
assert n3 < 0.5 * 33 and t3 < 0.5 * toks33, "...while using under half the samples"
assert pol[2][1] < pol[3][1] < pol[4][1], "a larger margin means more samples"
easy = [adaptive(i, 3)[1] for i, p in enumerate(SUITE) if bucket(p) == "arith 2-4"]; hard = [adaptive(i, 3)[1] for i, p in enumerate(SUITE) if bucket(p) == "arith 8-10"]
print(f"avg samples used: easy arithmetic {np.mean(easy):.1f}  vs  hard arithmetic {np.mean(hard):.1f}")
assert np.mean(hard) > np.mean(easy), "the policy spends compute where the answer is contested"

## 7 · When does thinking help? (and the API knobs)

| Task type | Extra thinking / samples help? | Why |
|---|---|---|
| multi-step arithmetic, algebra, planning, constraint puzzles | **Yes**, large | solution requires intermediate steps; errors can be caught |
| code generation with tests available | **Yes** (samples + run tests) | verifier is cheap and exact |
| ambiguous judgment calls (tone, taste) | little | no ground truth to converge on; voting averages style |
| retrieval/lookup, extraction, classification of clear cases | **No** | the bottleneck is knowledge/context, not reasoning; use RAG (A03) or a smaller model |
| latency-critical chat (< 1-2 s) | usually no | thinking tokens add sequential latency |
| tasks where the model has a *systematic* misconception | **No** (or worse) | samples are correlated; voting entrenches the error (section 8) |

**API knobs (reference: check current provider docs; names change).** Reasoning-capable models typically expose either an explicit *thinking token budget* or a coarser *effort* setting; the thinking tokens are billed as output tokens, may be returned (possibly summarised) for debugging, and count toward latency. In the Anthropic Messages API, extended thinking is enabled through a `thinking` parameter that carries a budget; consult the docs for the current parameters, minimums and which models support it. Do not hard-code model IDs or limits from a tutorial. A minimal illustrative (not executed) call looks like:

```python
# reference only - check current docs for parameter names / supported models
client.messages.create(model=MODEL, max_tokens=..., thinking={"type": "enabled", "budget_tokens": ...}, messages=[...])
```

**A routing rule of thumb that is easy to test:** try the cheap setting first; escalate to more thinking or more samples only when (a) a verifier fails, (b) samples disagree (vote margin small), or (c) a difficulty classifier says "hard". Section 6 *is* that policy. We encode the table above as a tiny helper so the decision is explicit and testable:

In [ ]:
def thinking_policy(has_verifier, multi_step, latency_ms_budget, systematic_error_risk=False):
    """Returns (thinking, samples). Illustrative thresholds, tune on your own eval."""
    if not multi_step: return ("minimal", 1)
    thinking = "minimal" if latency_ms_budget < 1500 else "standard"
    samples = 1
    if has_verifier: samples = 8                      # cheap exact check: sample-and-verify
    elif not systematic_error_risk and latency_ms_budget >= 5000: samples = 5     # vote only if errors look independent
    return (thinking, samples)
assert thinking_policy(False, False, 800) == ("minimal", 1)                       # FAQ lookup
assert thinking_policy(True, True, 30000) == ("standard", 8)                      # invoice maths with a calculator check
assert thinking_policy(False, True, 30000, systematic_error_risk=True) == ("standard", 1)
assert thinking_policy(False, True, 800) == ("minimal", 1)

## 8 · Break it
### 8a · Correlated errors: when the model is *consistently* wrong
Voting assumes samples err **independently**. Suppose 30% of questions contain a *trap* - a plausible shortcut that the model falls for most of the time (a systematic misconception, an ambiguous rounding rule). On trap questions the correct answer has $p=0.35$ and **one** specific wrong answer has probability 0.55. Voting then converges to the trap answer, not the truth: accuracy first rises (more samples help the non-trap questions), then *falls back* as the trap answer wins on the trap questions, and plateaus well below pass@N.

In [ ]:
rt = np.random.default_rng(11); PQ = 4000; trap = rt.uniform(size=PQ) < 0.30
def vote_with_traps(N):
    ok = 0.0
    p_ok = np.where(trap, 0.35, 0.80); p_trap_ans = np.where(trap, 0.55, 0.0); rest = 1 - p_ok - p_trap_ans
    probs = np.concatenate([p_ok[:, None], p_trap_ans[:, None], np.repeat((rest / 6)[:, None], 6, 1)], 1)
    C = multinomial_rows(rt, N, probs); mx = C.max(1); tied = (C == mx[:, None]).sum(1)
    return float(np.mean((C[:, 0] == mx) / tied))
passN = lambda N: float(np.mean(1 - (1 - np.where(trap, 0.35, 0.80)) ** N))
rows = [(N, vote_with_traps(N), passN(N)) for N in [1, 5, 17, 65, 257]]
for N, v, pn in rows: print(f"N={N:<4} majority {v:.3f}   pass@N {pn:.3f}")
plt.figure(figsize=(5.8, 3.4)); plt.semilogx([r[0] for r in rows], [r[1] for r in rows], "o-", label="majority vote"); plt.semilogx([r[0] for r in rows], [r[2] for r in rows], "k--", label="pass@N"); plt.legend(); plt.xlabel("N"); plt.title("systematic errors cap the vote"); plt.show()
assert rows[-1][1] < 0.75 and rows[-1][2] > 0.99, "pass@N -> 1 but the vote plateaus near 0.70 (= non-trap share) - the trap questions are lost"
assert rows[-1][1] - rows[-2][1] < 0.02, "adding samples no longer helps: a plateau, not a slow climb"
print("Fix: diversify (different prompts/models/temperature), add an independent verifier, or fix the misconception (data, tools, retrieval).")

### 8b · Cost without benefit
Thinking and sampling are billed whether or not they help. Using the same suite, compute the **marginal accuracy per extra 1,000 tokens** between the fixed-$N$ steps and find where it drops below a threshold you would pay for.

In [ ]:
tok = np.array(Nlist) * avg_tokens; acc = np.array(R["majority"])
marg = np.diff(acc) / (np.diff(tok) / 1000)
for (a, b), m in zip(zip(Nlist[:-1], Nlist[1:]), marg): print(f"N {a:>2} -> {b:<2}: +{m * 100:.2f} accuracy points per extra 1k tokens")
assert marg[0] > marg[-1], "diminishing returns"
worth = [(a, b) for (a, b), m in zip(zip(Nlist[:-1], Nlist[1:]), marg) if m * 100 >= 0.5]
print("steps worth paying for at a 0.5 pt per 1k tokens threshold:", worth)

## 9 · Exercises
**Practice (30 min)**
1. Using `maj_binary`, find the smallest odd $N$ for which a model with $p=0.7$ reaches $\ge99\%$ vote accuracy, and check it against the Hoeffding bound.
2. Extend `extract_answer` to accept `Answer: 1.2e3`, and write 3 assertions (including one that should *not* be normalised).

**Stretch (60 min)**
3. Change the ledger arithmetic error model so slips are *independent uniform integers* in $[-50,50]$. How do `majority@33` and `pass@33` change versus the clustered-slip version? Explain with section 2.
4. Add a third verifier: "format-only" (accepts any answer that parses). Show it does not beat `majority`.

**Challenge (2 h)**
5. Build a **two-stage router**: a cheap pass (N=3 votes); if the vote margin is 3-0 return, else escalate to N=17 with the program verifier. Report accuracy, average tokens and the fraction escalated, versus fixed N=9. Include an assertion that your router is on or above the fixed-$N$ cost/accuracy frontier from section 5.

In [ ]:
# Practice 1 starter - make the assertions pass
def smallest_odd_N(p, target):
    # TODO: search odd N upward until maj_binary(p, N) >= target
    raise NotImplementedError
# N = smallest_odd_N(0.7, 0.99)
# assert float(maj_binary(0.7, N)) >= 0.99 and float(maj_binary(0.7, N - 2)) < 0.99
# assert 1 - float(maj_binary(0.7, N)) <= math.exp(-2 * N * 0.2 ** 2)

## ✅ Checkpoint
1. A model is right 40% of the time on a binary yes/no question. Will 21-sample majority voting help? What if there are ten possible answers and the wrong ones are spread evenly?
2. Why can best-of-N with a learned verifier get *worse* as N grows, and what is the engineering response?
3. A product manager wants "maximum thinking" on every request. Give two reasons it can be the wrong default.
4. Your self-consistency accuracy plateaus at 0.72 while pass@64 is 0.96. What does the gap tell you, and what would you try?

<details><summary>Answers</summary>

1. Binary: no - voting *amplifies* the majority tendency, so with $p=0.4<0.5$ accuracy falls toward 0 as N grows (about 0.17 at N=21). With ten spread-out wrong answers a plurality is enough, so accuracy *rises* toward 1 (the correct answer, at 40%, beats each wrong one at about 6.7%).
2. More samples find more adversarial/"persuasive" wrong answers that exploit the verifier (the maximum score is an optimiser). Prefer programmatic verifiers, cap N using a gold eval, combine the verifier with voting, and monitor accuracy vs N.
3. Cost and latency scale with thinking tokens; much of the traffic (lookup, extraction, easy classification) is not reasoning-limited, so you pay for nothing; also past the saturation point more thinking yields flat or negative returns on some tasks.
4. Large gap = right answers are *reachable* but the vote/selection cannot find them: errors are correlated (systematic traps) or there is no good selector. Add an independent verifier/tests, diversify sampling (prompts, models), or fix the root cause with retrieval, tools, or fine-tuning (A05/A06).
</details>

## 📌 Takeaways
- Test-time compute is a third scaling axis: **width** (samples + aggregation), **depth** (longer reasoning) and **search**; each fixes a different bottleneck.
- Majority vote *amplifies the majority tendency*: it rescues $p>1/2$ (or spread-out wrong answers) and entrenches systematic errors; returns depend on the **distribution** of per-problem pass rates.
- Best-of-N is only as good as its verifier; a hackable verifier makes more samples *worse*. Programmatic checks beat learned judges where they exist.
- Give each chain enough budget to *reach* the answer, then buy reliability with samples; stop when the marginal accuracy per token is below what you would pay.
- Adaptive stopping (sample until the vote margin settles) keeps most of the accuracy for a fraction of the tokens.
- Always measure on your own eval: the curves here are simulations of *shapes*, not benchmark claims.

## 📚 Further reading
- Wang et al., *Self-Consistency Improves Chain of Thought Reasoning in Language Models* (2022), arXiv:2203.11171
- Wei et al., *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models* (2022), arXiv:2201.11903
- Cobbe et al., *Training Verifiers to Solve Math Word Problems* (2021), arXiv:2110.14168 (verifier-based best-of-N)
- Snell et al., *Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters* (2024), arXiv:2408.03314
- Provider documentation on extended thinking / effort settings (parameters change; read the current page)

**Next -> A08 · Multimodal** (vision, documents and audio inputs) - and revisit A04's eval discipline: every compute-for-accuracy trade in this lesson should be justified by a measured curve on your own data.